In [ ]:
import os

# Il percorso base dove Kaggle monta i dataset
dataset_path = '/kaggle/input'

In [ ]:

import os
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset

class MFNetDataset(Dataset):
    def __init__(self, base_path, transform=None):
        self.dir_termiche = os.path.join(base_path, "images")
        self.dir_maschere = os.path.join(base_path, "labels")
        self.dir_visual = os.path.join(base_path, "visual")
        
        # 1. Prendiamo i nomi dei file eliminando l'estensione (.png o .jpg)
        nomi_termiche = {os.path.splitext(f)[0] for f in os.listdir(self.dir_termiche)}
        nomi_maschere = {os.path.splitext(f)[0] for f in os.listdir(self.dir_maschere)}
        nomi_visual = {os.path.splitext(f)[0] for f in os.listdir(self.dir_visual)}
        
        # 2. Intersezione sui nomi "puri"
        self.nomi_comuni = sorted(list(nomi_termiche.intersection(nomi_maschere).intersection(nomi_visual)))
        
        print(f"Dataset inizializzato con successo: {len(self.nomi_comuni)} immagini comuni trovate.")
        self.transform = transform

    def __len__(self):
        return len(self.nomi_comuni)

    def __getitem__(self, idx):
        nome_base = self.nomi_comuni[idx]
        
        # 3. Costruiamo i percorsi usando l'estensione corretta per ogni cartella
        path_termica = os.path.join(self.dir_termiche, nome_base + ".png")
        path_maschera = os.path.join(self.dir_maschere, nome_base + ".png")
        path_visual = os.path.join(self.dir_visual, nome_base + ".jpg") # <--- NOTA IL .jpg
        
        image_termica = Image.open(path_termica).convert('RGB')
        image_maschera = Image.open(path_maschera)
        image_visual = Image.open(path_visual).convert('RGB')

        if self.transform:
            tensor_termica = self.transform(image_termica)
            tensor_visual = self.transform(image_visual)
        else:
            tensor_termica = image_termica
            tensor_visual = image_visual
            
        image_maschera = image_maschera.convert('L') # Forza la maschera a 1 canale (Grigio)
        mask_np = np.array(image_maschera)
        
        tensor_maschera = torch.as_tensor(mask_np, dtype=torch.long)

        # ORDINE FISSO: Termica, Visual, Maschera
        return tensor_termica, tensor_visual, tensor_maschera

In [ ]:
from torchvision import transforms

mia_trasformazione = transforms.ToTensor()

dataset_training = MFNetDataset(base_path="/kaggle/input/datasets/danialqashqai/mfnet-dataset/MFNet-dataset/ir_seg_dataset", transform=mia_trasformazione)

termica, maschera, visual = dataset_training[0]

print("Formato Termica:", type(termica), "- Dimensioni:", termica.shape)
print("Formato Maschera:", type(maschera), "- Dimensioni:", maschera.shape)
print("Formato visual:", type(visual), "- Dimensioni:", visual.shape)

In [ ]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        x = self.conv_block(x)
        return x


class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=9):
        super(UNet, self).__init__()
        
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.down1 = DoubleConv(in_channels, 64)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128, 256)
        self.down4 = DoubleConv(256, 512)
        
        self.bottleneck = DoubleConv(512, 1024)

    # IL DECODER
        
        self.up1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.up_conv1 = DoubleConv(1024, 512)
        
        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.up_conv2 = DoubleConv(512, 256)
        
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.up_conv3 = DoubleConv(256, 128)
        
        self.up4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.up_conv4 = DoubleConv(128, 64)
        
        self.final_conv = nn.Conv2d(64, num_classes, kernel_size=1)
        

    def forward(self, x):
        # --- 1. L'ENCODER (La discesa) ---
        x1 = self.down1(x)
        p1 = self.pool(x1)
        
        x2 = self.down2(p1)
        p2 = self.pool(x2)
        
        x3 = self.down3(p2)
        p3 = self.pool(x3)
        
        x4 = self.down4(p3)
        p4 = self.pool(x4)
        
        # --- 2. IL BOTTLENECK (Il fondo) ---
        b = self.bottleneck(p4)
        
        # --- 3. IL DECODER (La salita) ---
        
        # Step 1 di risalita
        u1 = self.up1(b)                             # Ingrandiamo il bottleneck
        concat1 = torch.cat([x4, u1], dim=1)         # Uniamo la skip connection (x4) con l'immagine ingrandita (u1)
        d1 = self.up_conv1(concat1)                  # Passiamo tutto nel DoubleConv
        
        # Step 2 di risalita
        u2 = self.up2(d1)
        concat2 = torch.cat([x3, u2], dim=1)
        d2 = self.up_conv2(concat2)
        
        # Step 3 di risalita (Tocca a te!)
        u3 = self.up3(d2)
        concat3 = torch.cat([x2, u3], dim=1)
        d3 = self.up_conv3(concat3)
        
        # Step 4 di risalita (Tocca a te!)
        u4 = self.up4(d3)
        concat4 = torch.cat([x1, u4], dim=1)
        d4 = self.up_conv4(concat4)
        
        # --- 4. OUTPUT FINALE ---
        out = self.final_conv(d4)
        
        return out

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader


# Diciamo a PyTorch di usare la GPU se disponibile, altrimenti la CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Sto usando: {device}")

# Creiamo il modello e lo spediamo sulla GPU
modello = UNet(in_channels=3, num_classes=9).to(device)
if torch.cuda.device_count() > 1:
    print(f"Sto usando {torch.cuda.device_count()} GPU!")
    modello = nn.DataParallel(modello)

# Il DataLoader raggruppa le immagini (es. 4 alla volta) e le mischia (shuffle)
dataloader = DataLoader(dataset_training, batch_size=4, shuffle=True, num_workers=2)

pesi = torch.tensor([0.1, 1.5, 2.0, 2.0, 1.0, 1.0, 1.0, 1.5, 1.5]).to(device)
criterio = nn.CrossEntropyLoss(weight=pesi)
# L'Ottimizzatore (Adam è il più famoso e stabile, lr è il 'Learning Rate' ovvero la velocità di apprendimento)
ottimizzatore = optim.Adam(modello.parameters(), lr=0.0001)


# --- 2. IL CICLO DI ADDESTRAMENTO ---

epoche = 15 # Quante volte guardiamo TUTTO il dataset
for epoca in range(epoche):
    modello.train() # Mette il modello in modalità "studente"
    
    # enumerate ci permette di avere il numero del batch (batch_idx) e i dati
    for batch_idx, (termica, visual, maschera) in enumerate(dataloader):
    
        # Spostiamo tutto sulla GPU
        termica = termica.to(device)
        visual = visual.to(device)
        maschera = maschera.to(device)
        
        # --- PROGETTO 1: Alleniamo sul TERMICO ---
        # Passiamo 'termica' al modello
        predizioni = modello(termica) 
        
        # Calcoliamo la loss usando la 'maschera' (quella vera!)
        loss = criterio(predizioni, maschera)
        
        # Step 3: Azzera i gradienti 
        ottimizzatore.zero_grad()
        
        # Step 4: Backward (Usa il metodo .backward() sulla tua 'loss')
        loss.backward()
        
        # Step 5: Aggiorna i pesi (Usa il metodo .step() sul tuo 'ottimizzatore')
        ottimizzatore.step()
        
        if batch_idx % 10 == 0:
            print(f"Epoca [{epoca+1}/{epoche}] - Batch {batch_idx} - Loss: {loss.item():.4f}")

In [ ]:
import matplotlib.pyplot as plt
import torch
import random
import numpy as np

# 1. Modalità Valutazione
modello.eval()

# 2. Peschiamo un indice a caso - ATTENZIONE ALL'ORDINE
indice_casuale = random.randint(0, len(dataset_training) - 1)
img_termica, img_visual, maschera_vera = dataset_training[indice_casuale]

# 3. Prepariamo l'input per il modello (Usiamo la TERMICÀ se stiamo testando il modello termico)
img_input = img_termica.unsqueeze(0).to(device)

# 4. Inferenza
with torch.no_grad():
    output = modello(img_input)
    predizione = torch.argmax(output, dim=1).squeeze(0)

# 5. Prepariamo le immagini per Matplotlib (portiamo tutto in formato HxWxC)
img_t_plot = img_termica.permute(1, 2, 0).cpu().numpy()
img_v_plot = img_visual.permute(1, 2, 0).cpu().numpy() 
mask_v_plot = maschera_vera.cpu().numpy()
pred_plot = predizione.cpu().numpy()

# --- 6. DISEGNIAMO IL CONFRONTO A 4 COLONNE ---
fig, assi = plt.subplots(1, 4, figsize=(20, 5))

# Immagine Visuale (RGB)
assi[0].imshow(img_v_plot)
assi[0].set_title("1. Immagine RGB")
assi[0].axis('off')

# Immagine Termica
assi[1].imshow(img_t_plot)
assi[1].set_title("2. Immagine Termica")
assi[1].axis('off')

# Maschera Reale
# Usiamo 'tab20' come colormap per distinguere bene le 9 classi
assi[2].imshow(mask_v_plot, cmap='tab20', vmin=0, vmax=8)
assi[2].set_title("3. Target (Verità)")
assi[2].axis('off')

# Predizione dell'AI
assi[3].imshow(pred_plot, cmap='tab20', vmin=0, vmax=8)
assi[3].set_title(f"4. Predizione AI (Epoche: {epoche})")
assi[3].axis('off')

plt.tight_layout()
plt.show()